# The QAOA part of the craiceann problem
## Base Problem
- The Problem Graph ($G_{ice}$): This graph represents a 3D volume. A surface crevasse can pass over a basal crack without touching it, but they are "connected" by the stress field between them. In graph theory, this "crossing without intersection" makes the graph non-planar. There is no way to draw it on a 2D sheet of paper without lines crossing.
- The Hardware Graph ($G_{chip}$): The quantum processor (the "chip") is strictly planar. The qubits are etched onto a 2D silicon wafer and only connect to their immediate neighbors (e.g., in a "Heavy Hex" lattice).

==> We must map the non-planar $G_{ice}$ onto the planar $G_{chip}$

### Workflow
#### 1. Building the "Problem Graph"
- use `rustworkx` (the high-performance graph library used by Qiskit) to construct a Weighted, Non-Planar Graph
- small enough to run on a laptop simulator but complex enough to demonstrate the 3D nature of the problem
#### 2. The Ising Mapping (Hamiltonian)
- [Jack]
#### 3. Transpilation / The "Embedding" Bottleneck
- use the Qiskit Transpiler to map the problem qubits to physical hardware qubits
- implement a SWAP Network to solve the embedding bottleneck
#### 4. QAOA Hybrid Loop
- start with a random sample ansatz
- create the QAOA Ansatz [$\gamma, \beta$]
- Loop -> Quantum Step, Classical Step, Update, Repeat
#### 5. Output / Prediction
- 



In [7]:
from qiskit import QuantumCircuit
from qiskit.circuit.library import QAOAAnsatz
from qiskit.transpiler import Layout
from qiskit_aer import AerSimulator

import rustworkx as rx
from rustworkx.visualization import mpl_draw as draw_graph

import numpy as np
import matplotlib.pyplot as plt

## 1. Building the "Problem Graph"

In [ ]:
def build_graph_model():
    """
    Builds a 2x3x2 Ice Shelf Graph (12 Nodes).
    Returns:
        graph (rustworkx.PyGraph): The weighted graph
        pos (dict): Positioning for plotting
        source_nodes (list): Indices of glacier anchor
        sink_nodes (list): Indices of ocean front
    """
    # TODO: Create a 2x3x2 grid graph
    graph = rx.PyGraph()
    
    

## 3. Transpilation / The "Embedding" Bottleneck

## 4. QAOA Loop

In [ ]:
layers = 2
qaoa_circuit = QAOAAnsatz(cost_operator=/*TODO: ADD-HAMILTONIAN*/, reps=layers)
qaoa_circuit.measure_all()
qaoa_circuit.draw("mpl")

# Transpile
pm = generate_preset_pass_manager(
    optimization_level=3, backend=generic_backend, seed_transpiler=seed
)

qaoa_circuit_transpiled = pm.run(qaoa_circuit)
qaoa_circuit_transpiled.draw("mpl", fold=False, idle_wires=False)

init_params = np.zeros(2 * layers)

objective_func_vals = []


def cost_func_estimator(
    params: list, ansatz: QuantumCircuit, isa_hamiltonian: SparsePauliOp, estimator: Estimator
) -> float:
    """Compute the cost function value using a parameterized ansatz and an estimator for a given Hamiltonian."""
    if isa_hamiltonian.num_qubits != ansatz.num_qubits:
        isa_hamiltonian = isa_hamiltonian.apply_layout(ansatz.layout)
    pub = (ansatz, isa_hamiltonian, params)
    job = estimator.run([pub])
    results = job.result()[0]
    cost = results.data.evs
    objective_func_vals.append(cost)
    return cost

def train_qaoa(
    params: list,
    circuit: QuantumCircuit,
    hamiltonian: SparsePauliOp,
    backend: QiskitRuntimeService.backend,
) -> tuple:
    """Optimize QAOA parameters using COBYLA and an estimator on a given backend."""
    with Session(backend=backend) as session:
        options = {"simulator": {"seed_simulator": seed}}
        estimator = Estimator(mode=session, options=options)
        estimator.options.default_shots = 100000

        result = minimize(
            cost_func_estimator,
            params,
            args=(circuit, hamiltonian, estimator),
            method="COBYLA",
            options={"maxiter": 200, "rhobeg": 1, "catol": 1e-3, "tol": 0.0001},
        )
    print(result)
    return result, objective_func_vals

def sample_qaoa(opt_params, circuit, backend):

    # Submit the circuit to Sampler
    options = {"simulator": {"seed_simulator": seed}}
    sampler = Sampler(mode=backend, options=options)
    job = sampler.run([(circuit, opt_params)], shots=SHOTS)
    results_sampler = job.result()
    counts_list = results_sampler[0].data.meas.get_counts()
    display(plot_histogram(counts_list, title=f"Max cut with {backend.name}"))

    return counts_list

result_qaoa, objective_func_vals = train_qaoa(
    init_params, qaoa_circuit_transpiled, cost_hamiltonian, generic_backend
)

NameError: name 'cost_hamiltonian' is not defined

In [ ]:
# Plot cost function
plt.figure(figsize=(12, 6))
plt.plot(objective_func_vals)
plt.xlabel("Iteration")
plt.ylabel("Cost")
plt.show()

In [ ]:
# Get the optimized parameters from the result
opt_params = result_qaoa.x
SHOTS = 10000
counts_list = sample_qaoa(opt_params, qaoa_circuit_transpiled, generic_backend)

## 5. Output / Prediction